# Тема 1. Тензоры и базовые операции
Тензор в PyTorch — это многомерный массив (как ndarray в NumPy), но с двумя ключевыми суперспособностями:
1) Он умеет производить вычисления на GPU (ускорение в десятки раз)
2) Он хранит историю операций для автоматического расчета градиентов (Autograd)


# Структура и архитектура PyTorch

PyTorch — это многослойная экосистема, состоящая из С++ ядра на нижнем уровне и удобных Python-модулей на верхнем.

---

## 1. Общая архитектура

### Основные компоненты ядра:
* **`torch.Tensor`** — Базовый многомерный массив. Поддерживает вычисления на GPU/TPU и хранит историю операций для расчета градиентов.
* **`torch.autograd`** — Движок автоматического дифференцирования. Строит динамический граф вычислений (Computational Graph) и вычисляет производные при вызове `.backward()`.
* **C++ Core (ATen & LibTorch)**:
  * **ATen**: Библиотека тензорных операций на C++.
  * **C10**: Нижний уровень абстракции для работы с памятью и устройствами.
  * **LibTorch**: C++ API для исполнения моделей без интерпретатора Python (для продакшена).

---

## 2. Ключевые модули Python API

### 🔹 `torch.nn` — Построение нейронных сетей
Содержит блоки для сборки архитектур:
* **`nn.Module`** — Базовый класс для всех моделей и слоев. Управляет параметрами (`.parameters()`) и переносом на устройства (`.to('cuda')`).
* **Слои**: `nn.Linear`, `nn.Conv2d`, `nn.LSTM`, `nn.Transformer` и др.
* **Функции потерь**: `nn.MSELoss`, `nn.CrossEntropyLoss` и др.

> **`torch.nn` vs `torch.nn.functional` (`F`)**:
> * `torch.nn` содержит слои со *состоянием* и обучемыми весами (объекты).
> * `torch.nn.functional` содержит *чистые функции* без состояния (например, `F.relu()`, `F.softmax()`).

---

### 🔹 `torch.optim` — Оптимизация
Модуль для обновления весов на основе градиентов:
* **Оптимизаторы**: `optim.SGD`, `optim.Adam`, `optim.AdamW`.
* **Планировщики (Schedulers)**: Управление изменением скорости обучения (Learning Rate).

---

### 🔹 `torch.utils.data` — Пайплайн данных
* **`Dataset`**: Абстрактный класс для определения логики получения единичного элемента (`__getitem__`) и размера датасета (`__len__`).
* **`DataLoader`**: Обертка над `Dataset` для батчевания, перемешивания (`shuffle`) и многопоточной загрузки (`num_workers`).

---

### 🔹 `torch.cuda` / `torch.mps` — Управление железом
Модули для контроля распределения памяти и взаимодействия с графическими ускорителями (NVIDIA CUDA, Apple Silicon MPS).

---

## 3. Сводная таблица модулей

| Модуль / Компонент | Назначение | Суть / Аналог |
| :--- | :--- | :--- |
| **`torch`** | Тензорные операции и математика | Аналог NumPy, но с поддержкой GPU |
| **`torch.nn`** | Слои, блоки и Loss-функции | ООП-конструктор нейросетей |
| **`torch.autograd`** | Расчет градиентов | Автоматическое дифференцирование |
| **`torch.optim`** | Оптимизаторы (Adam, SGD) | Алгоритмы обновления весов |
| **`torch.utils.data`** | `Dataset` и `DataLoader` | Загрузка и подготовка батчей |
| **`torch.jit`** | TorchScript | Экспорт и компиляция моделей |

---

## 4. Экосистема (Доменные библиотеки)

* **`torchvision`** — Компьютерное зрение (предобученные модели, аугментации, датасеты).
* **`torchaudio`** — Обработка аудио и спектрограмм.
* **`torchtext`** — Обработка текста и NLP.

---

## 5. Базовый цикл обучения (Training Loop)

```python
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Модель и оптимизатор
model = MyModel()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 2. Цикл обучения
for inputs, targets in dataloader:
    optimizer.zero_grad()          # Обнуляем градиенты
    outputs = model(inputs)        # Прямой проход (Forward pass)
    loss = criterion(outputs, targets) # Расчет ошибки
    loss.backward()                # Обратный проход (Autograd)
    optimizer.step()               # Обновление весов

## 1. Перенос тензоров на GPU
Чтобы код работал везде (и на GPU, и на CPU), в PyTorch принять динамически опеределять устройство:

In [1]:
import torch

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {device}')
if device.type == 'cuda':
    print(f'Название GPU: {torch.cuda.get_device_name(0)}')

Используется устройство: cuda
Название GPU: NVIDIA GeForce RTX 5060 Ti


## 2. Способы создания тензоров

In [2]:
# Создание из Python-списка или NumPy массива
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

# Тензоры со случайными значениями или нулями
zeros = torch.zeros(3, 3)
random_data = torch.randn(100, 784)  # 100 объектов, 784 признака

# Явное указание типа (float32 - стандарт для нейросетей)
x_float = torch.ones((2, 2), dtype=torch.float32)

print(random_data)

tensor([[-0.2769,  1.6160,  2.6104,  ..., -0.8257,  1.3018, -1.5557],
        [ 1.0203,  1.1016,  0.5687,  ..., -0.2204, -0.3351,  1.4489],
        [-0.1792, -0.3825, -1.6695,  ..., -0.9289, -1.8037, -1.1155],
        ...,
        [-0.4228, -0.8564, -0.2243,  ...,  0.7144, -1.2969,  0.9220],
        [ 1.1337, -0.5536, -0.8321,  ...,  0.9267, -2.0445,  1.0953],
        [ 1.5957, -0.2873,  0.4544,  ...,  0.6366, -0.9849, -0.3296]])


## 3. Перенос данных на устройство
Есть два основных способа отправить тензор на GPU:

In [3]:
# Способ 1: Использование .to(device) - самый универсальный и рекомендуемый!
data = torch.randn(64, 784)
data_gpu = data.to(device)

# Способ 2: Создание сразу на GPU
data_direct = torch.randn(64, 784, device=device)

print(f'Устройство тензора: {data_gpu.device}')

Устройство тензора: cuda:0


⚠️ Важное правило: Операции можно проводить только между тензорами, которые находятся на одном и том же устройстве! Если один тензор на CPU, а другой на GPU — PyTorch выдаст ошибку RuntimeError: Expected all tensors to be on the same device.

### 🎯 Задача Модуля 1: Нормализация и подготовка данных для GPU

**Контекст:** Вы получаете батч (пакет) изображений в виде сырого массива чисел с CPU, но для нейросети их нужно нормализовать, изменить форму и перенести на GPU.

**Задание:**
1. Определите устройство `device` (`cuda`, если GPU доступен, иначе `cpu`).
2. Создайте тензор `raw_images` размерности `(32, 28, 28)` (32 изображения 28x28) со случайными значениями от `0` до `255` типа `torch.float32` на CPU.
3. Нормализуйте значения тензора в диапазон от `0.0` до `1.0` (разделите на `255.0`).
4. Измените форму тензора на двумерный массив `(32, 784)` с помощью метода `.view()` или `.reshape()`.
5. Перенесите итоговый тензор на `device`.
6. Выведите на экран размерность (`.shape`), тип данных (`.dtype`) и устройство (`.device`) полученного тензора.

In [4]:
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
raw_images = torch.randint(low=0, high=256, size=(32, 28, 28), dtype=torch.float32, device='cpu')
raw_images_normalized = raw_images / 255
images = raw_images_normalized.reshape(32, 784)
images_gpu = images.to(device)
print(images_gpu.shape, images_gpu.dtype, images_gpu.device)

torch.Size([32, 784]) torch.float32 cuda:0


## Тема 2. Математические операции и Broadcasting
Нейросети на фундаментальном уровне — это комбинация матричных умножений, поэлементных операций и функций активации. Понимание того, как PyTorch работает с формами (shapes) и операциями над ними — ключ к эффективному коду.

### 1. Матричное умножение vs Поэлементное
В PyTorch важно не путать два вида умножения:

In [5]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[2.0, 0.0], [1.0, 2.0]])

# 1. Поэлементное умножение (Element-wise) — знаки * или torch.mul()
elem_wise = A * B
# Результат: [[1*2, 2*0], [3*1, 4*2]] -> [[2, 0], [3, 8]]

# 2. Матричное умножение (Matrix Multiplication) — оператор @ или torch.matmul()
matmul = A @ B
# Стандартное перемножение "строка на столбец"

⚠️ Главное правило матричного умножения: Внутренние размерности должны совпадать. Если $A$ имеет форму $(M, K)$, а $B$ имеет форму $(K, N)$, то результат $A @ B$ будет иметь форму $(M, N)$.

### 2. Broadcasting (Трансляция размерностей)
Broadcasting — это механизм, который позволяет PyTorch выполнять операции над тензорами разной размерности без явного копирования данных в памяти.

PyTorch автоматически "растягивает" меньший тензор вдоль недостающих осей по следующим правилам:

1) Размерности сравниваются справа налево.
2) Две размерности совместимы, если они равны, или одна из них равна 1.

Пример из реальной жизни: Прибавление вектора сдвигов (bias) к батчу признаков:

In [6]:
# Батч из 32 объектов, у каждого 64 признака
X = torch.randn(32, 64)

# Вектор смещений (bias) для 64 признаков
b = torch.randn(64)

# PyTorch сам растянет 'b' из формы (64,) до (1, 64),
# а затем применит к каждой из 32 строк!
result = X + b  # Форма результата: (32, 64)

### 🎯 Задача Модуля 2: Эмуляция линейного слоя с байасом и нормализацией

**Контекст:** Вы пишете свой первый полносвязный слой "вручную" через базовую алгебру PyTorch, используя матричное умножение, broadcasting и агрегацию.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Создайте тензор входов `X` размерности `(64, 100)` (батч из 64 объектов, 100 признаков) со случайными нормальными числами (`torch.randn`), сразу на `device`.
3. Создайте матрицу весов `W` размерности `(100, 10)` и вектор сдвигов (bias) `b` размерности `(10)` с помощью `torch.randn` на `device`.
4. Выполните прямое прохождение полносвязного слоя: $Y = X \cdot W + b$. *(Используйте матричное умножение и broadcasting!)*
5. Посчитайте вектор средних значений полученных выходов по каждому из 10 признаков вдоль батча (получится вектор формы `(10,)`).
6. Вычтите этот вектор средних из каждой строки тензора $Y$ (центрирование данных с помощью broadcasting).
7. Выведите форму (`.shape`) и устройство (`.device`) итогового тензора.

In [7]:
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X = torch.randn(size=(64, 100), device=device)
W = torch.randn(100, 10, device=device)
b = torch.randn(10, device=device)
Y = X @ W + b
means = torch.mean(Y, dim=0, keepdim=False)
Y -= means
print(Y.shape, Y.device)

torch.Size([64, 10]) cuda:0


## Модуль 3. Автоматическое дифференцирование (Autograd)
В основе обучения любых нейросетей лежит метод обратного распространения ошибки (Backpropagation). Чтобы не вычислять производные сложнейнших функций вручную, в PyTorch встроена система Autograd.

Когда ты выполняешь операции над тензорами, PyTorch на лету строит направленный ациклический граф вычислений. При вызове метода .backward() PyTorch автоматически проходит по этому графу назад и с помощью цепного правила считает градиенты всех параметров.

### 1. Флаг requires_grad=True
По умолчанию тензоры создаются с requires_grad=False. Если мы хотим, чтобы PyTorch отслеживал операции над тезорами и считал по нему производные (кк для весов модели), нужно указать requires_grad=True:

In [8]:
import torch


# Вес модели, по которому мы захотим посчитать градиент
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Входные данные (по ним градиент обычно не нужен)
x = torch.tensor([3.0])

# Прямой проход (Forward Pass): y = w * x + b
y = w * x + b

### Запуск .backward() и получение градиентов
Чтобы запустить вычисление градиентов, нужно вызывать метод .backward() у итоговой скалярной величины (обычно это функция потерь / Loss):

In [9]:
# Посчитаем простую функцию потерь: loss = y^2
loss = y ** 2

# Запускаем обратный проход (Backpropogation)
loss.backward()

# Градиенты d(loss)/dw и d(loss)/db автоматически сохранились в .grad
print(f"Градиент w: {w.grad}")  # d(loss)/dw = 2 * y * x = 2 * 7 * 3 = 42.0
print(f"Градиент b: {b.grad}")  # d(loss)/db = 2 * y * 1 = 2 * 7 * 1 = 14.0


Градиент w: tensor([42.])
Градиент b: tensor([14.])


### 3. Важные особенности Autograd
1) Накопление градиентов (Gradient Accumulation):
При повторном вызове .backward() градиенты складываются с предыдущими значениями! В реальном обучении перед каждым шагом оптимизации их зануляют: `w.grad.zero_()`.

2) Отключение градиентов (`torch.no_grad()`):
Во время валидации или тестирования модели нам не нужен граф вычислений — это экономит кучу памяти GPU и ускоряет работу:

In [10]:
with torch.no_grad():
    y_pred = w * x + b  # Операции внутри не строят граф вычислений

3) Отсоединение от графа (`.detach()`):
Создает новый тензор с теми же данными, но "отрезанный" от истории графа (у нового тензора `requires_grad=False`).

### 🎯 Задача Модуля 3: Ручной шаг градиентного спуска (Gradient Descent Step)

**Контекст:** Вы пишете базовый цикл обновления параметров нейросети «с нуля» без использования готовых оптимизаторов, с чистым Autograd и ручной очисткой градиентов.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Создайте обучаемый параметр `w` (скаляр или тензор формы `(1,)`) со значением `2.0`, указав `requires_grad=True` и отправив его на `device`.
3. Задайте константу скорости обучения `lr = 0.1` (learning rate).
4. Выполните 1-ю итерацию:
   - Вычислите функцию: $f(w) = w^2 - 4w + 4$.
   - Вызовите `.backward()` для вычисления производной $\frac{df}{dw}$.
   - Сохраните значение градиента `w.grad.item()` в переменную `grad1`.
   - Обновите параметр `w` вручную по формуле $w = w - lr \times \frac{df}{dw}$. *(💡 Внимание: обновление `w` делайте внутри контекста `with torch.no_grad():`, чтобы операция обновления не попала в граф вычислений!)*
   - Занулите градиент у `w` с помощью `w.grad.zero_()`.
5. Выполните 2-ю итерацию:
   - Снова вычислите функцию $f(w) = w^2 - 4w + 4$ с уже обновленным `w`.
   - Запустите `.backward()`.
   - Сохраните новый градиент `w.grad.item()` в переменную `grad2`.
6. Выведите на экран:
   - Значение `grad1` и обновленное `w` после 1-й итерации.
   - Значение `grad2` и значение функции $f(w)$ после 2-й итерации.

In [11]:
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
w = torch.tensor([2.0], requires_grad=True, device=device)
lr = 0.1

f = w ** 2 - 4 * w + 4
f.backward()
grad1 = w.grad.item()
with torch.no_grad():
    w -= lr * grad1
w.grad.zero_()

f = w ** 2 - 4 * w + 4
f.backward()
grad2 = w.grad.item()

print(grad1, w)
print(grad2, f)

0.0 tensor([2.], device='cuda:0', requires_grad=True)
0.0 tensor([0.], device='cuda:0', grad_fn=<AddBackward0>)


## Модуль 4: Построение нейросетей через `torch.nn`
В предыдущих модулях мы делали все вручную через сырые тензоры. Но при построении глубоких сетей ручная декларация матриц весов и байасов быстро превращается в хаос.

Модуль `torch.nn` предоставляет высокоуровневые строительные блоки: слои, функции активации, блоки регуляризации и удобные контейнеры.

### 1. Полносвязный слой (nn.Linear)
Слой `nn.Linear(in_features, out_features)` берет на себя создание матрицы весов `W` и вектора смещений `b`, а также автоматическую случайную инициализацию:

In [12]:
import torch
import torch.nn as nn


# Создаем слой: 784 входов (признаков), 64 выхода (скрытых нейрона)
linear_layer = nn.Linear(in_features=784, out_features=64)

# У слоя сразу есть собственные параметры (веса и байасы):
print(linear_layer.weight.shape)
print(linear_layer.bias.shape)

torch.Size([64, 784])
torch.Size([64])


### 2. Функции активации и Dropout
Для введения нелинейности используются функции активации (`nn.ReLU`, `nn.Sigmoid`, `nn.GELU`), а для защиты от переобучения - слой регуляризации `nn.Dropout(p)`:
* `nn.ReLU()`: Обнуляет все отрицательные значения: $f(x) = \max(0, x)$.
* `nn.Dropout(p=0.2)`: Во время обучения случайным образом с вероятностью $p$ "выключает" (зануляет) часть нейронов.

### 3. Контейнер `nn.Sequential`
`nn.Sequential` - это самый простой способ собрать цепь из слоев, через которые данные будут проходить последовательно друг за другом:

In [15]:
# Собираем простую сеть из 2-х слоев:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(128, 10) # 10 классов на выходе
)

# Переносим ВСЮ сетьна GPU в одну строчку!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Прогоняем батч данных через сеть
x = torch.randn(32, 784, device=device)
output = model(x) # Вызывает метод forward внутри Sequential
print(output.shape)

torch.Size([32, 10])


💡 Как устроен `add_module`?

В `nn.Sequential` слои можно добавлять не только разом в конструктор, но и по очереди через метод `.add_module(name, module)`. Имя модуля задается строкой, например:

In [16]:
seq = nn.Sequential()
seq.add_module('fc1', nn.Linear(784, 64))
seq.add_module('relu1', nn.ReLU())

### 🎯 Задача Модуля 4: Динамическое построение сети через `nn.Sequential` и `add_module`

**Контекст:** Вы подготавливаетесь к написанию кастомного класса `Perceptron`. Вам нужно научиться динамически в цикле собирать цепочку из слоев `Linear`, `ReLU` и `Dropout` с помощью метода `add_module`.

**Задание:**
1. Задайте `device` (`cuda`, если доступен, иначе `cpu`).
2. Задайте переменные параметров:
   - `input_dim = 100`
   - `hidden_dim = 32`
   - `output_dim = 2`
   - `num_layers = 3` (количество скрытых слоев)
   - `p = 0.2` (вероятность dropout)
3. Создайте пустой контейнер `model = nn.Sequential()`.
4. С помощью цикла `for i in range(num_layers)` динамически добавьте в `model` с помощью метода `.add_module()`:
   - Линейный слой `torch.nn.Linear(prev_size, hidden_dim)` с именем `'layer{}'.format(i)`
   - Активацию `torch.nn.ReLU()` с именем `'relu{}'.format(i)`
   - Дропаут `torch.nn.Dropout(p=p)` с именем `'dropout{}'.format(i)`
   - *(Не забудьте обновлять переменную `prev_size` на каждом шаге цикла!)*
5. После окончания цикла добавьте финальный классификационный слой `torch.nn.Linear(prev_size, output_dim)` с именем `'classifier'`.
6. Перенесите `model` на `device`.
7. Создайте случайный батч данных `X` формы `(16, input_dim)` на `device`.
8. Пропустите `X` через модель, получите `output` и выведите на экран:
   - Саму модель `print(model)` (PyTorch красиво напечатает структуру ее слоев).
   - Форму (`.shape`) и устройство (`.device`) полученного `output`.

In [18]:
import torch
import torch.nn as nn


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_dim = 100
hidden_dim = 32
output_dim = 2
num_layers = 3
p = 0.2

model = nn.Sequential()
prev_size = input_dim
for i in range(num_layers):
    model.add_module(f'layer{i+1}', nn.Linear(prev_size, hidden_dim))
    model.add_module(f'relu{i+1}', nn.ReLU())
    model.add_module(f'dropout{i+1}', nn.Dropout(p=p))
    prev_size = hidden_dim

model.add_module('classifier', nn.Linear(prev_size, output_dim))
model = model.to(device)

X = torch.randn(16, input_dim, device=device)
output = model(X)
print(model)
print(output.shape, output.device)

Sequential(
  (layer1): Linear(in_features=100, out_features=32, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (layer2): Linear(in_features=32, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.2, inplace=False)
  (layer3): Linear(in_features=32, out_features=32, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)
torch.Size([16, 2]) cuda:0


## Модуль 5. Кастомные модули (nn.Module) и продвинутые фишки
Теперь мы объединим абсолютно все знания, полученные за курс, чтобы разобрать наш целевой класс Perceptron до каждого символа.

### 1. Как устроен кастомный класс на базе nn.Module
Все нейросети в PyTorch наследуются от класса torch.nn.Module. У него есть два главных требования к структуре:
1) super().__init__() в конструкторе: ОБЯЗАТЕЛЬНО должен вызываться первым делом. Он инициализирует внутренюю инфраструктуру PyTorch (регистрацию слоев, параметров, буферов)
2) Метод forward(self, input): определяет, как данные проходят через слои. Когда ты пишешь model(X), PyTorch под капотом автоматически вызывает именно метод forward.

### 2. Продвинутый трюк: свойство @property def device


In [19]:
@property
def device(self):
    for p in self.parameters():
        return p.device

#### Зачем это нужно?
У самого класса torch.nn.Module по умолчанию нет прямого атрибута model.device. Если ты перенесешь модель на GPU через model.to(device), слои перейдут на GPU, но узнать устройство модели напрямую через model.device нельзя (вызовет AttributeError)

Этого геттер-свойство решает эту проблему:
1) self.parameters() возвращает генератор всех весов (nn.Parameter) модели.
2) for p in self.parameters(): return p.device берет самый первый вес сети и возврщает его .device.
3) Декоратор @property позволяет обращаться к этому методу как к обычному полю - model.device вместо model.device()